In [1]:
import os
import joblib

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from src.helpful_functions import check_if_all_numbers_present_for_column

# Metadata

In [2]:
proj_root = os.environ['PYTHONPATH']
proj_root

'C:\\Users\\aseva\\Desktop\\MyEDU\\OTUS\\ML_Adv\\OTUS_ML_Adv_GradProj_SaintSanych_v1'

# Data reading

## timestamps deanomymisation and working/none-working days

In [3]:
timestamps_df = pd.read_csv(os.path.join(proj_root, 'data', 'times', 'times_1_day.csv'))
timestamps_df['day'] = pd.to_datetime(timestamps_df['time']).map(lambda x: x.date())
timestamps_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 280 entries, 0 to 279
Data columns (total 3 columns):
 #   Column   Non-Null Count  Dtype 
---  ------   --------------  ----- 
 0   id_time  280 non-null    int64 
 1   time     280 non-null    str   
 2   day      280 non-null    object
dtypes: int64(1), object(1), str(1)
memory usage: 6.7+ KB


In [4]:
timestamps_df.duplicated().sum()

np.int64(0)

**`DataFrame`, видимо, не распознает тип `date` - проверим:**

In [5]:
type(timestamps_df.loc[0, 'day'])

datetime.date

### убедимся, что нумерация верно упорядочена

In [6]:
(timestamps_df.sort_values('id_time').index != timestamps_df.sort_values('day').index).sum() == 0

np.True_

### пришьем к нему информацию по рабочим дням

In [7]:
holidays_df = pd.read_csv(os.path.join(proj_root, 'data', 'weekends_and_holidays.csv'))
holidays_df['day'] = pd.to_datetime(holidays_df['Date']).map(lambda x: x.date())
holidays_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 91 entries, 0 to 90
Data columns (total 3 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   Date    91 non-null     str   
 1   Type    91 non-null     str   
 2   day     91 non-null     object
dtypes: object(1), str(2)
memory usage: 2.3+ KB


**перед мерджем набо проверить дубликаты:**

In [8]:
holidays_df.duplicated().sum()

np.int64(2)

**вот, блин! убираем:**

In [9]:
holidays_df.drop_duplicates(inplace=True)

In [10]:
ts_days_df = timestamps_df[['id_time', 'day']].merge(holidays_df[['Type', 'day']], on='day', how='left')
ts_days_df['week_no'] = ts_days_df['day'].map(lambda x: x.isocalendar().week)
ts_days_df['day_of_week'] = ts_days_df['day'].map(lambda x: x.weekday())
ts_days_df['is_working_day'] = ts_days_df['Type'].isna().map(int)
ts_days_df.drop(columns=['Type'], inplace=True)
ts_days_df.head(8)

,id_time,day,week_no,day_of_week,is_working_day
0,0,2023-10-09,41,0,1
1,1,2023-10-10,41,1,1
2,2,2023-10-11,41,2,1
3,3,2023-10-12,41,3,1
4,4,2023-10-13,41,4,1
5,5,2023-10-14,41,5,0
6,6,2023-10-15,41,6,0
7,7,2023-10-16,42,0,1


In [11]:
ts_days_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 280 entries, 0 to 279
Data columns (total 5 columns):
 #   Column          Non-Null Count  Dtype 
---  ------          --------------  ----- 
 0   id_time         280 non-null    int64 
 1   day             280 non-null    object
 2   week_no         280 non-null    int64 
 3   day_of_week     280 non-null    int64 
 4   is_working_day  280 non-null    int64 
dtypes: int64(4), object(1)
memory usage: 11.1+ KB


In [12]:
check_if_all_numbers_present_for_column(ts_days_df, 'id_time', True, True)

All numbers from 0 to 279 are present in column 'id_time'.


True

In [13]:
ts_days_df.to_csv(os.path.join(proj_root, 'data', 'ts_days_df.csv'), index=False)